# Практическая работа № 2. Работа с LLM через API

**Студент:** Архангельский Данила Андреевич  
**Группа:** ИСТ-05  
**Вариант:** 01 — первые 15 заявок из `rental_01.csv`


In [ ]:
# При необходимости установить зависимости:
# %pip install -r requirements.txt

import os, csv, json
from pathlib import Path
from typing import Optional

import pandas as pd
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from langchain_gigachat.chat_models import GigaChat
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate
from langchain_core.prompts import SystemMessagePromptTemplate, HumanMessagePromptTemplate
from langchain_core.output_parsers import StrOutputParser

load_dotenv()
API_KEY = os.getenv('GIGA_KEY')
if not API_KEY:
    raise RuntimeError('GIGA_KEY не найден. Создайте .env или добавьте переменную окружения.')

llm = GigaChat(credentials=API_KEY, model='GigaChat-2', verify_ssl_certs=False, temperature=0.2, max_tokens=1000)
print(llm.invoke('Ответь одним словом: работает?').content)


In [ ]:
VARIANT = '01'
def load_rental(path):
    rows = []
    with open(path, encoding='utf-8-sig', newline='') as f:
        reader = csv.reader(f, delimiter=';', quotechar='"')
        next(reader, None)
        for row in reader:
            if row:
                rows.append({'amount': int(row[0]), 'text': ';'.join(row[1:]).strip()})
    return pd.DataFrame(rows)

df = load_rental(Path('data') / f'rental_{VARIANT}.csv').head(15).copy().reset_index(drop=True)
assert len(df) == 15, f'Ожидалось 15 заявок, получено {len(df)}'
display(df)


## Часть 1. Базовый промпт и Few-shot


In [ ]:
basic_prompt = PromptTemplate(
    input_variables=['text'],
    template='Определи общее число проживающих, учитывай взрослых и детей, не считай животных. Если данных недостаточно, верни 0. Верни только целое число.\\nЗаявка: {text}\\nКоличество:'
)
basic_chain = basic_prompt | llm | StrOutputParser()

system = SystemMessagePromptTemplate.from_template(
    'Извлеки общее число проживающих. Считай взрослых и детей, не считай животных. '
    'Примеры: «пара»=2; «один взрослый и ребёнок»=2; «2 взрослых и 2 детей»=4; '
    '«с женой и двумя детьми»=4. Если однозначно определить нельзя, верни 0. Только целое число.'
)
human = HumanMessagePromptTemplate.from_template('Заявка: {text}\\nКоличество проживающих:')
few_chain = ChatPromptTemplate.from_messages([system, human]) | llm | StrOutputParser()

def run_chain(chain, texts):
    results = []
    for text in texts:
        try:
            results.append(chain.invoke({'text': text}))
        except Exception as exc:
            results.append(f'ERROR: {type(exc).__name__}: {exc}')
    return results

def to_num(series):
    return pd.to_numeric(series.astype(str).str.strip(), errors='coerce')

df['result_basic'] = run_chain(basic_chain, df.text)
df['result_fewshot'] = run_chain(few_chain, df.text)
for col in ['result_basic', 'result_fewshot']:
    pred = to_num(df[col])
    correct = int(pred.eq(df.amount).sum())
    print(f'{col}: {correct}/{len(df)}; ошибок {len(df)-correct}; точность {correct/len(df):.1%}')

errors = df.loc[
    to_num(df.result_basic).ne(df.amount) | to_num(df.result_fewshot).ne(df.amount),
    ['text', 'amount', 'result_basic', 'result_fewshot']
]
display(errors)


### Результаты части 1

На 15 заявках получено: базовый промпт — **6/15 (40,0%)**, Few-shot — **15/15 (100,0%)**. Результаты зафиксированы по фактическому запуску GigaChat.


## Часть 2. Структурированное извлечение JSON и Pydantic


In [ ]:
class RentalFields(BaseModel):
    count_adults: Optional[int] = None
    count_children: Optional[int] = None
    start_date: Optional[str] = None
    nights: Optional[int] = None
    price_per_day: Optional[int] = None
    remarks: Optional[str] = None

json_prompt = '''Ты извлекаешь структурированные данные из заявки на аренду жилья.
Верни только JSON с полями count_adults, count_children, start_date, nights, price_per_day, remarks.
Правила:
- Не придумывай отсутствующие сведения; неизвестные значения обозначай null.
- Животных не считай людьми.
- Если дано только общее количество людей, не разделяй его на взрослых и детей.
- start_date указывай в формате ДД.ММ.
- nights — количество ночей проживания.
- price_per_day — цена за сутки в рублях.
- remarks — дополнительные пожелания и условия.
Заявка: {text}
'''

def extract_json(text):
    response = llm.invoke(json_prompt.format(text=text)).content.strip()
    if response.startswith('```'):
        response = response.split('\\n', 1)[1].rsplit('```', 1)[0].strip()
    data = json.loads(response)
    return RentalFields.model_validate(data).model_dump()

predictions = []
for i, text in enumerate(df['text'], 1):
    try:
        predictions.append(extract_json(text))
        print(f'Заявка {i}/15: успешно')
    except Exception as exc:
        predictions.append(None)
        print(f'Заявка {i}/15: ошибка — {type(exc).__name__}: {exc}')

pred_df = pd.DataFrame([
    item if item is not None else {field: None for field in RentalFields.model_fields}
    for item in predictions
])
display(pred_df)


In [ ]:
# Сравнение с эталонной разметкой. Читаем значения как строки,
# чтобы pandas не преобразовывал даты вида 01.08 в числа.
gold = pd.read_csv('gold_fields_01.csv', dtype=str, keep_default_na=False)
assert len(gold) == 15 and len(pred_df) == 15

fields = ['count_adults', 'count_children', 'start_date', 'nights', 'price_per_day']
def normalize(value, field):
    if value is None or pd.isna(value):
        return None
    value = str(value).strip()
    if value.lower() in ('', 'nan', 'none', 'null'):
        return None
    if field == 'start_date':
        parts = value.split('.')
        if len(parts) == 2:
            return f'{int(parts[0]):02d}.{int(parts[1]):02d}'
    if field in ('count_adults', 'count_children', 'nights', 'price_per_day'):
        number = float(value)
        return int(number) if number.is_integer() else number
    return value

scores = {}
for field in fields:
    expected = [normalize(v, field) for v in gold[field]]
    actual = [normalize(v, field) for v in pred_df[field]]
    correct = sum(a == b for a, b in zip(actual, expected))
    scores[field] = correct / len(gold)
    mismatches = [i + 1 for i, (a, b) in enumerate(zip(actual, expected)) if a != b]
    print(f'{field}: {correct}/15 ({scores[field]:.1%}); ошибки в заявках: {mismatches}')

overall = sum(scores.values()) / len(scores)
print(f'\\nСредняя точность по пяти полям: {overall:.1%}')
results = pd.concat([gold[fields].add_prefix('gold_'), pred_df.add_prefix('pred_')], axis=1)
display(results)


## Выводы по части 2

Фактические результаты: количество взрослых — 13/15 (86,7%); количество детей — 15/15 (100,0%); дата заезда — 14/15 (93,3%); количество ночей — 10/15 (66,7%); цена за сутки — 15/15 (100,0%). Средняя точность по пяти полям — **89,3%**. Поле `remarks` сравнивалось содержательно и в числовую среднюю точность не включалось.

Наиболее сложным оказалось определение количества ночей. Для повышения надёжности вычисление длительности следует проверять отдельной программной функцией. Неизвестные сведения необходимо оставлять `null`, не додумывая их.


## Проверка характерных сценариев

1. **Явно указаны взрослые и дети:** например, «2 взрослых и 2 детей» — модель должна вернуть 2 и 2.
2. **Указано только общее число людей:** например, «4 человека» — нельзя автоматически относить всех к взрослым или детям; раздельные поля должны быть `null`.
3. **Неполная заявка с животными:** например, «один человек и две собаки» — человек учитывается как проживающий, собаки не учитываются; неизвестные дата, срок и цена остаются `null`.

Это анализ трёх характерных сценариев из постановки и выборки, а не отдельный автоматизированный набор unit-тестов.
